# Q1 — 웨이퍼 결함 분류 베이스라인 (4일 MVP)
전처리 완료된 `data/processed/*.npz`로 **ResNet18 파인튜닝 → 결과 확인**까지 한 번에.
위에서부터 셀을 순서대로 실행하세요. GPU 자동 사용.

In [ ]:
# --- 한글 폰트 + import ---
import sys
from pathlib import Path

# notebooks/quick 에서 실행 중이면 프로젝트 루트를 import path에 추가
_p = Path.cwd()
for cand in [_p, _p.parent, _p.parent.parent]:
    if (cand / "utils").is_dir():
        sys.path.insert(0, str(cand))
        PROJ = cand
        break
else:
    PROJ = _p

from utils.korean_font import set_korean_font
set_korean_font()

import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("프로젝트 루트:", PROJ)
print("PyTorch:", torch.__version__, "| CUDA 사용가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# --- 설정 (4일 MVP: 빠르게 결과 보는 게 목표) ---
CLASSES = ["Center","Donut","Edge-Loc","Edge-Ring","Loc","Near-full","Random","Scratch","none"]
NUM_CLASSES = len(CLASSES)

PROC = PROJ / "data" / "processed"
CKPT_DIR = PROJ / "models" / "checkpoints"
FIG_DIR  = PROJ / "results" / "figures"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 128
BATCH = 256          # GPU면 256 OK, OOM 나면 128/64로
EPOCHS = 6           # MVP라 짧게. 더 돌리고 싶으면 늘리기
LR = 3e-4
SEED = 42

torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", DEVICE, "| epochs:", EPOCHS, "| batch:", BATCH)

In [ ]:
# --- 데이터 로드 + 간략 EDA ---
def load_split(name):
    d = np.load(PROC / f"wafer_{name}.npz", allow_pickle=True)
    return d["X"], d["y"]

Xtr, ytr = load_split("train")
Xva, yva = load_split("val")
Xte, yte = load_split("test")
print("train:", Xtr.shape, "| val:", Xva.shape, "| test:", Xte.shape)
print("X dtype/범위:", Xtr.dtype, Xtr.min(), "~", Xtr.max(), "(0=배경,1=정상다이,2=불량다이)")

# 클래스 분포
counts = np.bincount(ytr, minlength=NUM_CLASSES)
fig, ax = plt.subplots(figsize=(9,4))
ax.barh(CLASSES, counts, color="steelblue")
for i,c in enumerate(counts):
    ax.text(c, i, f"  {c:,} ({c/counts.sum()*100:.1f}%)", va="center", fontsize=9)
ax.set_title("train 클래스 분포 — 'none'(정상)이 압도적, 심한 불균형")
ax.invert_yaxis()
plt.tight_layout(); plt.savefig(FIG_DIR/"q1_class_dist.png", dpi=110, bbox_inches="tight"); plt.show()

In [ ]:
# --- 클래스별 샘플 웨이퍼맵 ---
fig, axes = plt.subplots(2, 5, figsize=(14,6))
for k, ax in enumerate(axes.ravel()):
    if k >= NUM_CLASSES:
        ax.axis("off"); continue
    idx = np.where(ytr == k)[0][0]
    ax.imshow(Xtr[idx], cmap="viridis", vmin=0, vmax=2)
    ax.set_title(f"{CLASSES[k]}", fontsize=10); ax.axis("off")
plt.suptitle("클래스별 대표 웨이퍼맵", fontsize=13)
plt.tight_layout(); plt.savefig(FIG_DIR/"q1_samples.png", dpi=110, bbox_inches="tight"); plt.show()

In [ ]:
# --- Dataset + 증강 (웨이퍼맵은 회전/반전 불변) ---
class WaferDS(Dataset):
    def __init__(self, X, y, train=False):
        self.X = X; self.y = y; self.train = train
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        img = self.X[i].astype(np.float32) / 2.0   # 0/1/2 -> 0~1
        if self.train:
            k = np.random.randint(0, 4)             # 90도 단위 회전
            if k: img = np.rot90(img, k).copy()
            if np.random.rand() < 0.5: img = np.fliplr(img).copy()
            if np.random.rand() < 0.5: img = np.flipud(img).copy()
        img = (img - 0.5) / 0.5                      # 대략 [-1,1]
        img = np.expand_dims(img, 0).repeat(3, 0)    # 3채널 (pretrained resnet 입력)
        return torch.from_numpy(img), int(self.y[i])

# Windows/Jupyter 안정성 위해 num_workers=0 (데이터가 이미 RAM에 있어 충분히 빠름)
def make_loader(X, y, train):
    return DataLoader(WaferDS(X, y, train), batch_size=BATCH,
                      shuffle=train, num_workers=0, pin_memory=(DEVICE=="cuda"))

tr_loader = make_loader(Xtr, ytr, True)
va_loader = make_loader(Xva, yva, False)
te_loader = make_loader(Xte, yte, False)
print("배치 수 — train:", len(tr_loader), "val:", len(va_loader), "test:", len(te_loader))

In [ ]:
# --- 모델: ResNet18 파인튜닝 ---
from torchvision import models
try:
    net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
except Exception:
    net = models.resnet18(pretrained=True)   # 구버전 torchvision 호환
net.fc = nn.Linear(net.fc.in_features, NUM_CLASSES)
net = net.to(DEVICE)

# 클래스 불균형 대응: inverse-frequency 가중치 (상한 두어 폭주 방지)
w = counts.sum() / (NUM_CLASSES * np.maximum(counts, 1))
w = np.clip(w, 0, 20).astype(np.float32)
class_w = torch.tensor(w, device=DEVICE)
print("클래스 가중치:", {c: round(float(x),2) for c,x in zip(CLASSES, w)})

criterion = nn.CrossEntropyLoss(weight=class_w)
optimizer = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
# --- 학습 루프 (val macro-F1 기준 best 저장) ---
from sklearn.metrics import f1_score, accuracy_score
import time

@torch.no_grad()
def evaluate(loader):
    net.eval(); ys=[]; ps=[]
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        out = net(xb)
        ps.append(out.argmax(1).cpu().numpy()); ys.append(yb.numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    return accuracy_score(y,p), f1_score(y,p,average="macro"), y, p

hist = {"loss":[], "val_acc":[], "val_f1":[]}
best_f1 = -1; best_path = CKPT_DIR / "baseline_resnet18.pt"

for ep in range(1, EPOCHS+1):
    net.train(); t0=time.time(); running=0.0
    for bi,(xb,yb) in enumerate(tr_loader):
        xb,yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(net(xb), yb)
        loss.backward(); optimizer.step()
        running += loss.item()
        if bi % 100 == 0:
            print(f"  ep{ep} batch {bi}/{len(tr_loader)} loss={loss.item():.3f}", end="\r")
    scheduler.step()
    tr_loss = running/len(tr_loader)
    va_acc, va_f1, _, _ = evaluate(va_loader)
    hist["loss"].append(tr_loss); hist["val_acc"].append(va_acc); hist["val_f1"].append(va_f1)
    flag = ""
    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save({"model": net.state_dict(), "classes": CLASSES, "img_size": IMG_SIZE}, best_path)
        flag = "  <- best 저장"
    print(f"[ep{ep}/{EPOCHS}] loss={tr_loss:.3f} | val_acc={va_acc:.3f} | val_macroF1={va_f1:.3f} | {time.time()-t0:.0f}s{flag}")

print("\nbest val macro-F1:", round(best_f1,4), "->", best_path)

In [ ]:
# --- 학습 곡선 ---
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(hist["loss"], marker="o"); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(hist["val_acc"], marker="o", label="val_acc")
ax[1].plot(hist["val_f1"], marker="s", label="val_macroF1")
ax[1].set_title("검증 성능"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig(FIG_DIR/"q1_curves.png", dpi=110, bbox_inches="tight"); plt.show()

In [ ]:
# --- test 평가: confusion matrix + classification report ---
from sklearn.metrics import classification_report, confusion_matrix
import itertools

# best 체크포인트 로드 후 평가
ck = torch.load(best_path, map_location=DEVICE)
net.load_state_dict(ck["model"])
te_acc, te_f1, yt, pt = evaluate(te_loader)
print(f"TEST  accuracy={te_acc:.3f} | macro-F1={te_f1:.3f}\n")
print(classification_report(yt, pt, target_names=CLASSES, digits=3, zero_division=0))

cm = confusion_matrix(yt, pt)
cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(8,7))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("예측"); ax.set_ylabel("실제"); ax.set_title("정규화 혼동행렬 (행=실제)")
for i,j in itertools.product(range(NUM_CLASSES), range(NUM_CLASSES)):
    ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center",
            color="white" if cmn[i,j]>0.5 else "black", fontsize=8)
fig.colorbar(im, fraction=0.046)
plt.tight_layout(); plt.savefig(FIG_DIR/"q1_confusion.png", dpi=110, bbox_inches="tight"); plt.show()

## 정리 & 다음 디벨롭 포인트

이 노트북으로 **베이스라인 결과**가 나왔습니다. `none`이 85%라 accuracy는 높게 보이지만, **macro-F1 / 클래스별 recall**이 진짜 성능 지표예요.

디벨롭 후보 (결과 보고 우선순위 정하기):
- `Near-full`(105장), `Donut`(389장) 같은 소수 클래스 → 오버샘플링 / focal loss
- 증강 강화, 이미지 64px로 줄여 더 빠른 실험 사이클
- `none` vs defect 2단계 분류 (이상 탐지 → 세부 분류)
- 결함 원인 멀티태스크 헤드 (configs/mappings/defect_to_cause.yaml)

다음: **Q2 노트북**에서 Grad-CAM으로 모델 근거 확인 + 단일 웨이퍼 추론 데모.